**This notebook is an exercise in the [Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning) course.  You can reference the tutorial at [this link](https://www.kaggle.com/alexisbcook/data-leakage).**

---


Most people find target leakage very tricky until they've thought about it for a long time.

So, before trying to think about leakage in the housing price example, we'll go through a few examples in other applications. Things will feel more familiar once you come back to a question about house prices.

# Setup

The questions below will give you feedback on your answers. Run the following cell to set up the feedback system.

In [2]:
# Set up code checking
from learntools.core import binder
binder.bind(globals())
from learntools.ml_intermediate.ex7 import *
print("Setup Complete")

Setup Complete


# Step 1: The Data Science of Shoelaces

Nike has hired you as a data science consultant to help them save money on shoe materials. Your first assignment is to review a model one of their employees built to predict how many shoelaces they'll need each month. The features going into the machine learning model include:
- The current month (January, February, etc)
- Advertising expenditures in the previous month
- Various macroeconomic features (like the unemployment rate) as of the beginning of the current month
- The amount of leather they ended up using in the current month

The results show the model is almost perfectly accurate if you include the feature about how much leather they used. But it is only moderately accurate if you leave that feature out. You realize this is because the amount of leather they use is a perfect indicator of how many shoes they produce, which in turn tells you how many shoelaces they need.

Do you think the _leather used_ feature constitutes a source of data leakage? If your answer is "it depends," what does it depend on?

After you have thought about your answer, check it against the solution below.

Highly likely.

- Target leakage: They just cannot know in advance of "The amount of leather they ended up using in the current month". Hence this information cannot be provided to make a decision for this matter.

In [3]:
# Check your answer (Run this code cell to receive credit!)
q_1.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct:</span> 

This is tricky, and it depends on details of how data is collected (which is common when thinking about leakage). Would you at the beginning of the month decide how much leather will be used that month? If so, this is ok. But if that is determined during the month, you would not have access to it when you make the prediction. If you have a guess at the beginning of the month, and it is subsequently changed during the month, the actual amount used during the month cannot be used as a feature (because it causes leakage).

Wait... you said that "Would you at the beginning of the month decide how much leather will be used that month? If so, this is ok."

... something's weird about this one, but I can't quite pinpoint that...

### Takeaway

Source of my *instinctive discomfort*:

    Decision / Control variables (different beast) vs. Objective / External variables.

<!-- Assuming that the **purpose** of making this model is like

    H0: forcasting: “How many laces will we need next month?”

instead of 

    H1: "Operational model": “Given planned production, how many laces do we need?” -->

Given that "you at the beginning of the month decide how much leather will be used that month",

- *leather used*: Control Variables 👉 These are chosen by the company
- Others (Month, Unemployment rate, Past advertising): Objective Variables 👉 These are given by the world

My discomfort arises from the fact that

    `Control Variables` has more inconsistency in population 'distribution' than `Objective Variables`.

vis. the **relationship** between control variables and the target variable is *WAY* more likely to be changed between the training time and the predicting time, because the relationship depends on the way such variables are created.

Hence I personally recommend not to put such variables *especially* for forcasting purpose e.g. “How many laces will we need next month?”

# Step 2: Return of the Shoelaces

You have a new idea. You could use the amount of leather Nike ordered (rather than the amount they actually used) leading up to a given month as a predictor in your shoelace model.

Does this change your answer about whether there is a leakage problem? If you answer "it depends," what does it depend on?

### My Answer

Nope, still the same answer; it depends. If you are guaranteed to have that data before prediction then it is just fine (which seems so if I interpret this as 'the amount of the leather Nike ordered in the beginning of this month')

SIDENOTE: but it seems so 'unrelated'... Not having any 'simple-and-consistent relationship' with the target.

Note that here, 

- 'simple' relationship: This tells 'high correlation' between some two random variables.
- 'consistent' relationship: This tells that the 'meaningful change of relationship' ('meaningful' enough to affect the pragmatic performance and even the usefulness of the model) is 'less likely to happen'

GPT:

- "simple" => strong signal
- "consistent" => stable relationship


In [3]:
# Check your answer (Run this code cell to receive credit!)
q_2.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct:</span> 

This could be fine, but it depends on whether they order shoelaces first or leather first. If they order shoelaces first, you won't know how much leather they've ordered when you predict their shoelace needs. If they order leather first, then you'll have that number available when you place your shoelace order, and you should be ok.

# Step 3: Getting Rich With Cryptocurrencies?

You saved Nike so much money that they gave you a bonus. Congratulations.

Your friend, who is also a data scientist, says he has built a model that will let you turn your bonus into millions of dollars. Specifically, his model predicts the price of a new cryptocurrency (like Bitcoin, but a newer one) one day ahead of the moment of prediction. His plan is to purchase the cryptocurrency whenever the model says the price of the currency (in dollars) is about to go up.

The most important features in his model are:
- Current price of the currency
- Amount of the currency sold in the last 24 hours
- Change in the currency price in the last 24 hours
- Change in the currency price in the last 1 hour
- Number of new tweets in the last 24 hours that mention the currency

The value of the cryptocurrency in dollars has fluctuated up and down by over $\$$100 in the last year, and yet his model's average error is less than $\$$1. He says this is proof his model is accurate, and you should invest with him, buying the currency whenever the model says it is about to go up.

Is he right? If there is a problem with his model, what is it?

They likely did:

random train/test split

instead of:

time-based split (past → future)

Let's assume that:

1. The problem the model attempts to solve is this:
   - input (assume its time is $t$ sec.s after epoch):
     
        - $P_t$ : Current price of the currency
        - $X_{sold}$: #(number of) currency sold during the last 24 hrs
        - $P_t-P_{t-86400}$
        - $P_t-P_{t-3600}$
        - $X_{tweets}$: Number of new tweets in the last 24 hours that mention the currency
   - output:
       - Expected price $\widehat{P_{t+86400}}$: Price of the cryptocurrency after 24 hrs.

I'm not so sure on how the validation process is held, so I'd first check it.

But from my intuition, it would have some problem... the 'best possible forcasting model' having such features might not give 'enough information' to reach for like, \\$ 1 mean absolute error by tick for even a one day. (e.g. # tweets could 'highly correlate' to the 'absolute value of the slope of future price changes' instinctively, than 'inc/dec', and the bitcoin price itself is way too much varying ( \\$ 100 as mentioned) than the error) Hence this seems to be caused from some data leakage, and I'd be skeptical to have faith about the model to invest my money.


In [4]:
# Check your answer (Run this code cell to receive credit!)
q_3.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct:</span> 

There is no source of leakage here. These features should be available at the moment you want to make a predition, and they're unlikely to be changed in the training data after the prediction target is determined. But, the way he describes accuracy could be misleading if you aren't careful. If the price moves gradually, today's price will be an accurate predictor of tomorrow's price, but it may not tell you whether it's a good time to invest. For instance, if it is $100 today, a model predicting a price of $100 tomorrow may seem accurate, even if it can't tell you whether the price is going up or down from the current price. A better prediction target would be the change in price over the next day. If you can consistently predict whether the price is about to go up or down (and by how much), you may have a winning investment opportunity.

### ChatGPT-ed

Mine

1. The validation error turned out to be 'unrealistically small'.
2. We could infer that there had been some data leakage. ==> He may not be right.
3. To support #2, if the validation set splitting is not done without regarding that those are series data and that the columns are constructed related to time series random variable (?), this kinda stuff may happen because the training set could contain some information for the validation set

---

ChatGPT

1. The validation error is unrealistically small.
2. This strongly suggests data leakage or an evaluation flaw.
3. A likely cause is improper validation (e.g., random splitting in time series), where training data contains information too close in time to the validation data, effectively leaking future information.


# Step 4: Preventing Infections

An agency that provides healthcare wants to predict which patients from a rare surgery are at risk of infection, so it can alert the nurses to be especially careful when following up with those patients.

You want to build a model. Each row in the modeling dataset will be a single patient who received the surgery, and the prediction target will be whether they got an infection.

Some surgeons may do the procedure in a manner that raises or lowers the risk of infection. But how can you best incorporate the surgeon information into the model?

You have a clever idea. 
1. Take all surgeries by each surgeon and calculate the infection rate among those surgeons.
2. For each patient in the data, find out who the surgeon was and plug in that surgeon's average infection rate as a feature.

Does this pose any target leakage issues?
Does it pose any train-test contamination issues?

An agency that provides healthcare wants to predict which patients from a rare surgery are at risk of infection, so it can alert the nurses to be especially careful when following up with those patients.

You want to build a model. Each row in the modeling dataset will be a single patient who received the surgery, and the prediction target will be whether they got an infection.

Some surgeons may do the procedure in a manner that raises or lowers the risk of infection. But how can you best incorporate the surgeon information into the model?

You have a clever idea. 
1. Take all surgeries by each surgeon and calculate the infection rate among those surgeons.
2. For each patient in the data, find out who the surgeon was and plug in that surgeon's average infection rate as a feature.

Does this pose any target leakage issues?
Does it pose any train-test contamination issues?

---

So to clarify in my own language, problem to solve:

Former Problem:
    - Input:
        - Surgery-got patient information
    - Output:
        - whether they got an infection

Wanted: Some surgeons may do the procedure in a manner that raises or lowers the risk of infection. But how can you best incorporate the surgeon information into the model?

Idea: Add a feature (to input) s.t.
    - Map each patient to the surgeon who executed the operation
    - And for each surgeon map it finally to that surgeon's *average infection rate*

Questions:

1. Does this pose any target leakage issues?
2. Does it pose any train-test contamination issues (In my term, I'd say *the case where validation data get involved into model pre-training before validation*)?

---

My understanding of the concepts:

- **Target leakage**: *the use of information during model training that would not be available at prediction time* (https://en.wikipedia.org/wiki/Leakage_(machine_learning))
- **train-test contamination**: *the case where validation data influences (including indirectly) the model pre-validation training*

---

Well, let's find out...

- Training Time: Data available for the original input and output and the *average infection rate* until that moment.
- Prediction Time: *average infection rate* cannot be renewed ...

My answers seems to depend on the exact meaning of *average infection rate* by its usages.

To answer the first question, I'd say yes if the feature *average infection rate* is meant to consider the recent infection rate after each operation (which I think it is). Then it **A1. does pose the target leakage issue**: *average infection rate* we already put for training uses its own target value at the time of prediction. A necessary condition where this would hold false is to *neglect the case of infection-from-patient after training*.

To answer the second question, this also depends... It does pose a train-test contamination issue if you did not use the *average infection rate* we put is based on the whole train + validation data, but it does not if you only used the constants based solely on the training set's such value for pre-validation training.

As a sidenote, you *could* make such idea not having target-leakage and train-test contamination free: Just use the *average infection rate* ONLY from the training data for pre-validation train and from the training & validation data for post-validation final training.

In [3]:
# Check your answer (Run this code cell to receive credit!)
q_4.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct:</span> 

This poses a risk of both target leakage and train-test contamination (though you may be able to avoid both if you are careful).

You have target leakage if a given patient's outcome contributes to the infection rate for his surgeon, which is then plugged back into the prediction model for whether that patient becomes infected. You can avoid target leakage if you calculate the surgeon's infection rate by using only the surgeries before the patient we are predicting for. Calculating this for each surgery in your training data may be a little tricky.

You also have a train-test contamination problem if you calculate this using all surgeries a surgeon performed, including those from the test-set. The result would be that your model could look very accurate on the test set, even if it wouldn't generalize well to new patients after the model is deployed. This would happen because the surgeon-risk feature accounts for data in the test set. Test sets exist to estimate how the model will do when seeing new data. So this contamination defeats the purpose of the test set.

# Step 5: Housing Prices

You will build a model to predict housing prices.  The model will be deployed on an ongoing basis, to predict the price of a new house when a description is added to a website.  Here are four features that could be used as predictors.
1. Size of the house (in square meters)
2. Average sales price of homes in the same neighborhood
3. Latitude and longitude of the house
4. Whether the house has a basement

You have historic data to train and validate the model.

Which of the features is most likely to be a source of leakage?

In [ ]:
# Fill in the line below with one of 1, 2, 3 or 4.
potential_leakage_feature = ____

# Check your answer
q_5.check()

In [ ]:
#q_5.hint()
#q_5.solution()

# Conclusion
Leakage is a hard and subtle issue. You should be proud if you picked up on the issues in these examples.

Now you have the tools to make highly accurate models, and pick up on the most difficult practical problems that arise with applying these models to solve real problems.

There is still a lot of room to build knowledge and experience. Try out a [Competition](https://www.kaggle.com/competitions) or look through our [Datasets](https://kaggle.com/datasets) to practice your new skills.

Again, Congratulations!

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/intermediate-machine-learning/discussion) to chat with other learners.*